<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/solutions/10_dqn_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# R10 DQN:Q学習をニューラルネットで行う(notebook)

本文:[`10-dqn.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/rl/10-dqn.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

> **これは解答つきの版である。** まず演習の版(`rl/10_dqn.ipynb`)を自分で解き、答え合わせに使うこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

# CartPole(R9 と同じ。Gymnasium の CartPole-v1 と同じ運動方程式)
class CartPole:
    gravity, masscart, masspole, length, force_mag, tau = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02
    total_mass = masscart + masspole
    polemass_length = masspole * length
    theta_limit, x_limit = 12 * 2 * np.pi / 360, 2.4
    def __init__(self, max_steps=500):
        self.max_steps = max_steps
        self.rng = np.random.default_rng()
    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.state = self.rng.uniform(-0.05, 0.05, 4)
        self.t = 0
        return self.state.copy(), {}
    def step(self, action):
        x, x_dot, th, th_dot = self.state
        force = self.force_mag if action == 1 else -self.force_mag
        c, s = np.cos(th), np.sin(th)
        temp = (force + self.polemass_length * th_dot**2 * s) / self.total_mass
        th_acc = (self.gravity * s - c * temp) / (self.length * (4.0 / 3.0 - self.masspole * c**2 / self.total_mass))
        x_acc = temp - self.polemass_length * th_acc * c / self.total_mass
        x, x_dot = x + self.tau * x_dot, x_dot + self.tau * x_acc
        th, th_dot = th + self.tau * th_dot, th_dot + self.tau * th_acc
        self.state = np.array([x, x_dot, th, th_dot])
        self.t += 1
        terminated = bool(abs(x) > self.x_limit or abs(th) > self.theta_limit)
        truncated = (not terminated) and self.t >= self.max_steps
        return self.state.copy(), 1.0, terminated, truncated, {}

# D1 で作った多層パーセプトロン(ReLU)と、勾配法の一種 Adam(このまま使う)
def init_params(sizes, rng):
    return [(rng.normal(0, np.sqrt(2 / m), (m, n)), np.zeros(n)) for m, n in zip(sizes[:-1], sizes[1:])]

def forward(params, X):
    hs, h = [X], X                       # hs:各層の出力(逆伝播で使う)
    for i, (W, b) in enumerate(params):
        z = h @ W + b
        h = np.maximum(z, 0) if i < len(params) - 1 else z    # 最後の層は活性化なし(価値は負にもなる)
        hs.append(h)
    return h, hs

def backward(params, hs, dout):
    grads, d = [], dout                  # dout:損失を出力で微分したもの
    for i in reversed(range(len(params))):
        W, b = params[i]
        grads.append((hs[i].T @ d, d.sum(axis=0)))
        if i > 0:
            d = (d @ W.T) * (hs[i] > 0)
    return grads[::-1]

def copy_params(params):
    return [(W.copy(), b.copy()) for W, b in params]

class Adam:
    def __init__(self, params, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0
        self.m = [[np.zeros_like(W), np.zeros_like(b)] for W, b in params]
        self.v = [[np.zeros_like(W), np.zeros_like(b)] for W, b in params]
    def step(self, params, grads):
        self.t += 1
        new = []
        for i, (p, g) in enumerate(zip(params, grads)):
            pair = []
            for j in range(2):
                self.m[i][j] = self.b1 * self.m[i][j] + (1 - self.b1) * g[j]
                self.v[i][j] = self.b2 * self.v[i][j] + (1 - self.b2) * g[j] ** 2
                m_hat = self.m[i][j] / (1 - self.b1 ** self.t)
                v_hat = self.v[i][j] / (1 - self.b2 ** self.t)
                pair.append(p[j] - self.lr * m_hat / (np.sqrt(v_hat) + self.eps))
            new.append(tuple(pair))
        return new

## 0. 準備:Q ネットワーク

CartPole(R9)の状態(4つ)を入力し、2つの行動の価値を出力するネット(4 → 64 → 64 → 2)を使う。ネットの関数は D1 で作ったものを用意してある。念のため、逆伝播の勾配を数値微分で確かめておく。

In [ ]:
rng = np.random.default_rng(0)
net = init_params([4, 8, 8, 2], rng)
X = rng.normal(size=(5, 4)); dout = rng.normal(size=(5, 2))
loss = lambda p: np.sum(forward(p, X)[0] * dout)          # 出力と dout の内積:出力での微分が dout になる
out, hs = forward(net, X)
g = backward(net, hs, dout)
W0, b0 = net[0]; h = 1e-6
num = np.zeros_like(W0)
for i in range(W0.shape[0]):
    for j in range(W0.shape[1]):
        Wp, Wm = W0.copy(), W0.copy(); Wp[i, j] += h; Wm[i, j] -= h
        num[i, j] = (loss([(Wp, b0)] + net[1:]) - loss([(Wm, b0)] + net[1:])) / (2 * h)
check("1層目の重みの勾配 = 数値微分", g[0][0], num, tol=1e-6)

## 1. replay buffer

経験 `(s, a, r, s2, d)` を、決まった大きさの配列にためる。いっぱいになったら、古いものから上書きする(リングバッファ)。

**実装**:`sample` で、ためた経験の中から `batch_size` 個を**一様にランダムに**選ぶ(本文 §4)。

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, rng):
        self.s, self.a = np.zeros((capacity, 4)), np.zeros(capacity, dtype=int)
        self.r, self.s2, self.d = np.zeros(capacity), np.zeros((capacity, 4)), np.zeros(capacity)
        self.capacity, self.size, self.ptr, self.rng = capacity, 0, 0, rng

    def push(self, s, a, r, s2, d):
        i = self.ptr
        self.s[i], self.a[i], self.r[i], self.s2[i], self.d[i] = s, a, r, s2, d
        self.ptr = (self.ptr + 1) % self.capacity              # いっぱいになったら先頭に戻って上書きする
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = self.rng.integers(self.size, size=batch_size)    # 一様にランダムに選ぶ
        return self.s[idx], self.a[idx], self.r[idx], self.s2[idx], self.d[idx]

In [ ]:
buf = ReplayBuffer(10, np.random.default_rng(0))
for k in range(25):
    buf.push(np.full(4, k), k % 2, float(k), np.full(4, k + 1), 0.0)
check("大きさ 10 のバッファに 25 個入れると、残るのは最後の 10 個", np.sort(buf.r), np.arange(15, 25))
s_b, a_b, r_b, s2_b, d_b = buf.sample(1000)
check("取り出した経験は、すべてバッファの中のもの", np.isin(r_b, np.arange(15, 25)).all(), True)
check("10 個がほぼ同じ回数ずつ選ばれる(各 100 回前後)", np.abs(np.bincount(r_b.astype(int) - 15) - 100).max() < 40, True)

## 2. TD目標

本文の式 (3):$y = r + \gamma (1 - d) \max_b Q_{\theta^-}(s', b)$ を、ミニバッチ全体で一度に計算する。

**手計算**:本文の確認課題1を解く。

- 手計算:(ここに書く)

In [ ]:
def td_targets(target_params, r, s2, d, gamma):
    q_next = forward(target_params, s2)[0].max(axis=1)
    return r + gamma * (1 - d) * q_next

In [ ]:
# 確認課題1:目標ネットの出力が (12, 15) になるように、最後の層だけのネットを作って確かめる
fixed = [(np.zeros((4, 2)), np.array([12.0, 15.0]))]
check("d = 0 なら 1 + 0.99 * 15", td_targets(fixed, np.array([1.0]), np.zeros((1, 4)), np.array([0.0]), 0.99), [15.85])
check("d = 1 なら r だけ", td_targets(fixed, np.array([1.0]), np.zeros((1, 4)), np.array([1.0]), 0.99), [1.0])

## 3. 1回の学習と、ε の減らし方

1回の学習(`train_step`)は、次の順に行う。出力のうち、勾配を流すのは選んだ行動 `a` の出力だけである(本文 §2)。誤差は ±1 で切る(大きな誤差で重みが急に変わらないようにする工夫で、Huber 損失を使うことにあたる)。

ε は、 1.0 から 0.05 まで、`decay_steps` ステップかけて直線的に減らす(本文 §5)。

**実装**:`epsilon_by_step` を書く。`step >= decay_steps` では 0.05 のままにする。

In [ ]:
def train_step(params, target_params, opt, buffer, batch_size=64, gamma=0.99):
    s, a, r, s2, d = buffer.sample(batch_size)
    y = td_targets(target_params, r, s2, d, gamma)            # 「正解」として扱い、微分しない
    out, hs = forward(params, s)
    err = np.clip(out[np.arange(batch_size), a] - y, -1.0, 1.0)
    dout = np.zeros_like(out)
    dout[np.arange(batch_size), a] = err / batch_size         # 選んだ行動の出力にだけ勾配を流す
    return opt.step(params, backward(params, hs, dout))

def epsilon_by_step(step, eps_start=1.0, eps_end=0.05, decay_steps=5000):
    frac = min(step / decay_steps, 1.0)
    return eps_start + frac * (eps_end - eps_start)

In [ ]:
check("epsilon の減り方", [epsilon_by_step(t) for t in [0, 2500, 5000, 9000]], [1.0, 0.525, 0.05, 0.05])

## 4. DQN で CartPole を学ぶ

**予想**:R9 の REINFORCE と比べて、DQN の学習曲線は、なめらかか、でこぼこか。

- 予想:(ここに書く)

1回の学習に 10 秒前後かかる。

In [ ]:
def run_dqn(n_episodes=300, capacity=50000, target_every=500, lr=1e-3, gamma=0.99, batch_size=64, seed=0):
    rng = np.random.default_rng(seed)
    env = CartPole(); env.reset(seed=seed)
    params = init_params([4, 64, 64, 2], rng)
    target_params = copy_params(params)
    opt, buffer = Adam(params, lr), ReplayBuffer(capacity, rng)
    history, step = [], 0
    for ep in range(n_episodes):
        s, _ = env.reset(); done, ep_return = False, 0.0
        while not done:
            eps = epsilon_by_step(step)
            a = int(rng.integers(2)) if rng.random() < eps else int(forward(params, s[None])[0].argmax())
            s2, r, terminated, truncated, _ = env.step(a)
            buffer.push(s, a, r, s2, float(terminated))       # d には terminated だけを入れる
            if buffer.size >= batch_size:
                params = train_step(params, target_params, opt, buffer, batch_size, gamma)
            if step % target_every == 0:
                target_params = copy_params(params)            # 目標ネットワークにコピーする
            s, done, ep_return, step = s2, terminated or truncated, ep_return + r, step + 1
        history.append(ep_return)
    return params, np.array(history)

def moving_average(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode="valid")

params, hist = run_dqn(seed=0)
plt.plot(hist, alpha=0.3, label="episode return"); plt.plot(moving_average(hist), label="moving average 20")
plt.xlabel("episode"); plt.ylabel("return"); plt.legend(); plt.title("DQN on CartPole"); plt.show()
print(f"最初の50エピソードの平均 {hist[:50].mean():.1f} → 最後の50エピソードの平均 {hist[-50:].mean():.1f}")

In [ ]:
def greedy_eval(params, n=10):
    env = CartPole(); rets = []
    for k in range(n):
        s, _ = env.reset(seed=1000 + k); done, R = False, 0.0
        while not done:
            s, r, terminated, truncated, _ = env.step(int(forward(params, s[None])[0].argmax()))
            R += r; done = terminated or truncated
        rets.append(R)
    return np.array(rets)

ev = greedy_eval(params)
print("学習後に greedy(ε = 0)で動かした収益(10回):", ev)
check("ランダムな方策(平均 20 程度)よりずっと長く立つ", ev.mean() > 100, True)

## 5. 工夫を取り除く

本文 §4 の表を確かめる。seed を2つずつ使い、次の3つを比べる(1分ほどかかる)。

- 標準:バッファ 50000、目標ネットワークは 500 ステップごとにコピー
- バッファを小さく:大きさ 500(最近の経験だけで学ぶのに近い)
- 目標ネットワークなし:毎ステップコピー(目標がいつも動く)

**予想**:どれがいちばん悪くなるか。

- 予想:(ここに書く)

In [ ]:
settings = {"standard": dict(), "small buffer (500)": dict(capacity=500), "no target net (copy every step)": dict(target_every=1)}
curves = {}
for name, kw in settings.items():
    curves[name] = np.array([run_dqn(seed=k, **kw)[1] for k in range(2)])
    print(f"{name:32s}: 最後の100エピソードの平均 {curves[name][:, -100:].mean(axis=1).round(1)}(seed ごと)")
for name, c in curves.items():
    plt.plot(moving_average(c.mean(axis=0)), label=name)
plt.xlabel("episode"); plt.ylabel("return (2-seed mean, moving average 20)"); plt.legend(); plt.show()

結果は seed によって大きく変わる。2つの seed だけでは、はっきりした結論は言えないことに注意する(本文 §6)。研究では、もっと多くの seed で比べる。

## 6. 研究で使うときは

`cpai_lab` には DQN は入っていない(本文 §7)。研究では、PyTorch で書かれたライブラリの DQN を使うことが多い。この notebook の `capacity`、`batch_size`、`target_every`、`epsilon_by_step` の引数、`lr`、`gamma` が、ライブラリの設定のどれにあたるかを、本文 §7 の表で確かめておく。

## 7. 説明(自分の言葉で書く)

1. DQN の損失 (2) で、TD目標 $y$ を微分しないのはなぜか
   - (ここに書く)
2. replay buffer と目標ネットワークは、それぞれ何を防いでいるか。§5 の結果と合わせて説明する
   - (ここに書く)
3. バッファに入れる $d$ に `truncated` を入れてはいけないのはなぜか
   - (ここに書く)